# Sesión 10 · Potencia, riesgo y responsabilidad técnica  
## Notebook de seguimiento (versión sobresaliente)

> Este notebook entrena **criterio profesional**.  
> No busca “ganar accuracy”, sino **evitar errores caros** al usar frameworks de Deep Learning.

### Qué vas a practicar aquí
1. **Declarar una red explícitamente** (nada de magia).
2. Ejecutar un **Check de Cordura** antes de confiar en el entrenamiento.
3. Identificar **errores silenciosos** (el código corre, pero el modelo está mal).
4. Entrenar con **trazabilidad** (callbacks).
5. Hacer **inferencia real** (ver números crudos).


---
## 0 · Preparación del entorno


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)


---
## 1 · Dataset base (controlado, pero no trivial)

Usamos un dataset sintético **para aislar el efecto de tus decisiones** (arquitectura, escala, entrenamiento).  
No es un dataset “de juguete” (hay ruido y variables redundantes).

> Recuerda: en proyectos reales, la dificultad no es “entrenar”, sino **diagnosticar**.


In [ ]:

X, y = make_classification(
    n_samples=4000,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    weights=[0.6, 0.4],
    class_sep=1.0,
    flip_y=0.03,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Fraude (clase 1) rate en train:", round(y_train.mean(), 4))


---
## 2 · Escalado (decisión explícita)

En Keras, **no escalar** suele ser un error silencioso:
- convergencia errática,
- gradientes inestables,
- aprendizaje lentísimo o inexistente.

Aquí escalamos de forma explícita.


In [ ]:

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


---
## 3 · Arquitectura mínima consciente (sin optimizar)

Declaramos una red mínima para clasificación binaria:
- 1 capa oculta (ReLU)
- salida Sigmoid (probabilidad)

> Keras no te da mejores modelos. Te da **más palancas**.


In [ ]:

def build_good_model(input_dim: int) -> Sequential:
    model = Sequential([
        Dense(10, activation="relu", input_shape=(input_dim,)),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

model = build_good_model(X_train_s.shape[1])
model.summary()


---
## 4 · Check de Cordura (Sanity Check) — ANTES de entrenar

Este check no evalúa si el modelo es bueno.  
Evalúa si el modelo **tiene derecho a ser evaluado**.

### 4.1 ¿La salida tiene el rango correcto?
Si la salida es `sigmoid`, debe estar en **[0, 1]**.

### 4.2 ¿La pérdida inicial tiene sentido?
En clasificación binaria con entropía cruzada, con pesos aleatorios, la pérdida inicial suele estar cerca de:

> **≈ 0.69**

**Reflexión forzada**
- Si tu pérdida inicial es ~0.69 → el modelo empieza en “azar” (bien).
- Si tu pérdida inicial es 15.0 → algo está roto (datos, escala, arquitectura o pérdida).


In [ ]:

# 4.1 Rango de salida (sin entrenar)
y_pred_init = model.predict(X_train_s[:256], verbose=0)
print("Rango de salida inicial:", float(y_pred_init.min()), float(y_pred_init.max()))

# 4.2 Pérdida inicial (sin entrenar)
init_loss, init_acc = model.evaluate(X_train_s[:1024], y_train[:1024], verbose=0)
print("Pérdida inicial (sin entrenar):", round(float(init_loss), 4))
print("Acc inicial (sin entrenar):", round(float(init_acc), 4))


🧠 **Interpretación rápida** (escríbela aquí en comentarios)

- ¿Tu pérdida inicial está cerca de 0.69?
- Si no lo está: ¿qué sospechas primero (escala, activación, loss, shape)?


In [ ]:

# TU RESPUESTA (comentarios):
# - ...


---
## 5 · Catálogo de errores silenciosos (experimentos controlados)

En Keras puede ocurrir algo peligroso:
> el código corre, pero el modelo está técnicamente muerto.

Vamos a provocar **dos fallos** típicos:
1) Activación incorrecta en la salida  
2) Entrenar sin escalar

**Objetivo:** que aprendas a detectar el fallo antes de mirar métricas finales.


### 5.1 Error silencioso: activación incorrecta en la salida (ReLU)

Esto rompe el significado probabilístico de la salida en clasificación binaria.
El modelo puede “entrenar”, pero lo que produce no son probabilidades.


In [ ]:

def build_bad_output_activation(input_dim: int) -> Sequential:
    model = Sequential([
        Dense(10, activation="relu", input_shape=(input_dim,)),
        Dense(1, activation="relu")  # ❌ incorrecto para probabilidad binaria
    ])
    # Mantener binary_crossentropy adrede para que se vea el absurdo (puede dar training raro)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

bad_act_model = build_bad_output_activation(X_train_s.shape[1])

# Sanity check rápido
bad_pred = bad_act_model.predict(X_train_s[:256], verbose=0)
bad_loss, bad_acc = bad_act_model.evaluate(X_train_s[:1024], y_train[:1024], verbose=0)

print("Rango salida (bad activation):", float(bad_pred.min()), float(bad_pred.max()))
print("Pérdida inicial (bad activation):", round(float(bad_loss), 4))


### Mini-tarea (obligatoria)
1) ¿Qué parte del sanity check te grita que esto está mal?  
2) ¿Por qué este fallo es “silencioso”?

Responde en comentarios.


In [ ]:

# TU RESPUESTA:
# 1) ...
# 2) ...


### 5.2 Error silencioso: entrenar sin escalar (comparación breve)

Aquí **NO escalamos** a propósito.  
No buscamos precisión, buscamos observar comportamiento (curva/loss).


In [ ]:

# Misma arquitectura, pero entrenando con datos NO escalados
model_no_scale = build_good_model(X_train.shape[1])

# Check de cordura (sin entrenar) en NO escalado
pred_ns = model_no_scale.predict(X_train[:256], verbose=0)
loss_ns, acc_ns = model_no_scale.evaluate(X_train[:1024], y_train[:1024], verbose=0)

print("Rango salida inicial (no scale):", float(pred_ns.min()), float(pred_ns.max()))
print("Pérdida inicial (no scale):", round(float(loss_ns), 4))


In [ ]:

history_ns = model_no_scale.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    verbose=0
)

plt.plot(history_ns.history["loss"], label="train (no scale)")
plt.plot(history_ns.history["val_loss"], label="val (no scale)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Entrenamiento SIN escalado (observación)")
plt.show()


### Mini-tarea (obligatoria)
Compara mentalmente (no con números finos) lo que esperas de:
- entrenamiento sin escalar (este)
- entrenamiento escalado (el siguiente bloque)

¿Qué riesgo profesional hay si “te da pereza” escalar? Responde en comentarios.


In [ ]:

# TU RESPUESTA:
# ...


---
## 6 · Entrenamiento con trazabilidad (callbacks)

Un profesional **no se queda mirando la pantalla**.  
Configura el entrenamiento para:
- detenerse cuando deja de mejorar,
- guardar automáticamente el mejor modelo.

Usaremos:
- `EarlyStopping`
- `ModelCheckpoint`


In [ ]:

model = build_good_model(X_train_s.shape[1])

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

ckpt_path = "/mnt/data/sesion10_best_model.keras"
checkpoint = ModelCheckpoint(
    filepath=ckpt_path,
    monitor="val_loss",
    save_best_only=True,
    verbose=0
)

history = model.fit(
    X_train_s, y_train,
    validation_split=0.2,
    epochs=60,
    callbacks=[early_stop, checkpoint],
    verbose=0
)

print("Épocas entrenadas:", len(history.history["loss"]))
print("Mejor val_loss:", round(float(np.min(history.history["val_loss"])), 4))


In [ ]:

plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Curvas con trazabilidad (EarlyStopping + Checkpoint)")
plt.show()


---
## 7 · Evaluación (después de coherencia + trazabilidad)

Solo ahora miramos métricas.

> Cambiar de herramienta no cambia las reglas del juego.


In [ ]:

y_prob = model.predict(X_test_s, verbose=0).ravel()
y_hat = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_hat)
print("Accuracy test:", round(float(acc), 4))


---
## 8 · Inferencia real (bajar a tierra la abstracción)

Aquí está la celda que falta en casi todos los cursos:  
ver **números crudos**.

Objetivo:
- ¿son probabilidades?
- ¿están en [0, 1]?
- ¿están cerca de 0.5 cuando el modelo duda?


In [ ]:

sample = X_test_s[:5]
probs_5 = model.predict(sample, verbose=0).ravel()
preds_5 = (probs_5 >= 0.5).astype(int)

print("Probabilidades (5 casos):", np.round(probs_5, 4))
print("Predicciones (umbral 0.5):", preds_5)
print("Etiquetas reales:", y_test[:5])


### Mini-tarea (obligatoria)
1) ¿Ves probabilidades “dudosas” cerca de 0.5? ¿Qué significan?  
2) ¿Qué cambia si el umbral fuera 0.7? (No lo calcules aún: razona.)

Responde en comentarios.


In [ ]:

# TU RESPUESTA:
# 1) ...
# 2) ...


---
## 9 · Nota de producción: coste de inferencia (mensaje, no benchmark)

- Un modelo clásico en Scikit-Learn suele ser más ligero y rápido en CPU.
- Un modelo de Keras puede ser más pesado (memoria/latencia), y suele justificar GPU solo si el problema lo pide.

> La potencia de Keras tiene un coste. ¿Tu sistema puede permitírselo?


---
## 10 · Preguntas de cierre (evaluable)

Responde en este notebook:

1) ¿Qué dos señales del **Check de Cordura** mirarías siempre antes de entrenar?  
2) Describe un error silencioso que **no rompe Python**, pero invalida el modelo.  
3) En un proyecto real tabular de 30.000 filas, ¿qué condiciones deberían cumplirse para justificar Keras?  
4) ¿Qué aporta EarlyStopping a tu responsabilidad técnica? (no repitas “evita overfitting”: explica el *por qué*).  


In [ ]:

# RESPUESTAS:
# 1) ...
# 2) ...
# 3) ...
# 4) ...


---
## Cierre del notebook

Este cuaderno no busca que “uses Keras”.  
Busca que **no lo uses a la ligera**.

> La diferencia entre un modelo académico  
> y un sistema profesional  
> no es la librería,  
> es el criterio con el que se usa.
